In [ ]:
## Vloco de codigo para instalar versao especifica dos pacotes
##pip install pandas==1.5.3
##pip install numpy==1.24.3

# Carregando pacotes

In [ ]:
# Pacotes de manipulacao
import sys
import os
import pandas as pd
#import numpy as np

# Pacotes de visualizacao
#import matplotlib.pyplot as plt
#import seaborn as sns

#print("pandas:", pd.__version__)
#print("numpy:", np.__version__)

# Conexão ao repositório via gdrive

In [ ]:

# Acesso aos módulos do diretório Colab x Google Drive
from google.colab import drive
drive.mount('/content/gdrive')

pasta_in = 'base_score_bureau_movel_full/'
pasta_out = 'Feature_store/'
path_padrao = "/content/gdrive/Othercomputers/Meu laptop"

bucket_trusted = f"{path_padrao}/Database_trusted/{pasta_in}"
bucket_feature_store =f"{path_padrao}/{pasta_out}"

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


# Conexão ao repositório via OCI

In [ ]:
# Buckets e nomes de saída nuvem = "oci://"
#pasta_in = 'base_score_bureau_movel_full/'
#pasta_out = 'Feature_store/'
#namespace = "@grxzqsiaote6/"
#bucket_trusted = f"oci://TRUSTED{namespace}{pasta_in}"
#bucket_feature_store = f"oci://BOOKS_VARIAVEIS{namespace}{pasta_out}"

# Carregando databases

#### Base Dados Cadastrais

In [20]:
## Carregando todos arquivos em parquet de uma pasta
#path = project_root +'database/raw/base_score_bureau_movel/base_score_bureau_movel/'

#all_files = [os.path.join('database/raw/base_score_bureau_movel/', f) for f in os.listdir(project_root/'database/raw/base_score_bureau_movel/') if f.endswith('.parquet')]
#df_list = [pd.read_parquet(f, engine='pyarrow') for f in all_files]
#df_bureau = pd.concat(df_list, ignore_index=True)

df_bureau = pd.read_parquet(bucket_trusted, engine='pyarrow')
df_bureau.head()

,ts_proc,Ano,Mes,FLAG_INSTALACAO,ProductDescription,ProductMigration,SCORE_01,SCORE_02,FPD,NUM_CPF,SAFRA
0,20260309213950,2024,10,True,CMV,Aquisição,2.0,1.0,1.0,ZZZZZZZX7T9,202410
1,20260309213950,2024,10,False,CMV,None,562.0,559.0,NaN,ZZZZZZZ8TZ8,202410
2,20260309213950,2024,10,False,CMV,None,585.0,559.0,NaN,ZZZZZZW9XWN,202410
3,20260309213950,2024,10,True,CMV,PRE,562.0,636.0,0.0,ZZZZZX7XWY8,202410
4,20260309213950,2024,10,True,CMV,Aquisição,538.0,570.0,1.0,ZZZZZX8TTUZ,202410


In [ ]:
df_bureau.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3795310 entries, 0 to 3795309
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ts_proc             object 
 1   Ano                 int64  
 2   Mes                 int64  
 3   FLAG_INSTALACAO     bool   
 4   ProductDescription  object 
 5   ProductMigration    object 
 6   SCORE_01            float32
 7   SCORE_02            float32
 8   FPD                 Int64  
 9   NUM_CPF             object 
 10  SAFRA               int64  
dtypes: Int64(1), bool(1), float32(2), int64(3), object(4)
memory usage: 267.8+ MB


#### Ajustando os tipos de dados

In [21]:
# Iremos ajustar os tipos de dados para otimizar a memoria e o correto processamento
df_bureau['Ano'] = df_bureau['Ano'].astype('int')
df_bureau['Mes'] = df_bureau['Mes'].astype('int')
df_bureau['FLAG_INSTALACAO'] = df_bureau['FLAG_INSTALACAO'].astype('bool')
df_bureau['ProductDescription'] = df_bureau['ProductDescription'].astype('object') #PROD
df_bureau['ProductMigration'] = df_bureau['ProductMigration'].astype('object') #flag_mig2
df_bureau['SCORE_01'] = df_bureau['SCORE_01'].astype('float32')
df_bureau['SCORE_02'] = df_bureau['SCORE_02'].astype('float32')
df_bureau['FPD'] = df_bureau['FPD'].astype('Int64')
df_bureau['NUM_CPF'] = df_bureau['NUM_CPF'].astype('object')
df_bureau['SAFRA'] = df_bureau['SAFRA'].astype('int')

#### Feature Engineer

Iremos criar apenas as variáveis:
- Relação entre `SCORE_01` e `SCORE_02`
- Média entre `SCORE_01` e `SCORE_02`
- Diferenca entre `SCORE_01` e `SCORE_02`
- Minino entre `SCORE_01` e `SCORE_02`

In [22]:
# Iremos criar as medidas descritas acima
df_bureau['SCORE_RATE'] = df_bureau['SCORE_02'] / df_bureau['SCORE_01']
df_bureau['SCORE_AVG'] = (df_bureau['SCORE_01'] + df_bureau['SCORE_02']) / 2
df_bureau['SCORE_DIFF'] = df_bureau['SCORE_02'] - df_bureau['SCORE_01']
df_bureau['SCORE_MIN'] = df_bureau[['SCORE_01', 'SCORE_02']].min(axis=1)

In [23]:
bucket_feature_store

'/content/gdrive/Othercomputers/Meu laptop/Feature_store/'

In [ ]:
# Salvando o dataframe em parquet
df_bureau.to_parquet(f"{bucket_feature_store}/book_variaveis_01.parquet", index=False)